In [10]:
from setup import *

## Metadaten für alle RIS laden

Entities sind die angebundenen RIS. Mit diesem Notebook downloaden wir alle entities, die in Poliscope aktuell vorlagen, inklusive Metadaten und ids.

Diese Liste ist nötig, um später gezielt für bestimmte Kommunen Daten zu finden, oder Rechercheergebnisse passend zu filtern.


In [11]:
# Alle Items sammeln
all_entities = []

# Paginierungsparameter
#achtung, tatsächlich ist gerade auf 300 Limit
limit = 500
offset = 0
total = 16079

print("/entities")

# Durch alle Pages iterieren
while offset < total:
    response = poliscope_request(
        "GET",
        "/entities",
        params={
            "limit": limit,
            "offset": offset,
            "detail": "standard",
        },
        timeout=10.0,
    )
    data = response.json()
    items = data.get("data", [])
    all_entities.extend(items)  # Alle Items zur Liste hinzufügen
    
    print(f"Downloaded {offset + len(items)} / {total} items")
    offset += limit

/entities
Downloaded 500 / 16079 items
Downloaded 1000 / 16079 items
Downloaded 1500 / 16079 items
Downloaded 2000 / 16079 items
Downloaded 2500 / 16079 items
Downloaded 3000 / 16079 items
Downloaded 3500 / 16079 items
Downloaded 4000 / 16079 items
Downloaded 4500 / 16079 items
Downloaded 5000 / 16079 items
Downloaded 5500 / 16079 items
Downloaded 6000 / 16079 items
Downloaded 6500 / 16079 items
Downloaded 7000 / 16079 items
Downloaded 7500 / 16079 items
Downloaded 8000 / 16079 items
Downloaded 8500 / 16079 items
Downloaded 9000 / 16079 items


KeyboardInterrupt: 

In [ ]:
entities_df = pd.DataFrame(all_entities)
entities_df

In [ ]:
entities_df.to_csv("./data/metadata/all_entities.csv", index=False)

# Großstädte filtern

Basierend auf dem entities-dataframe können wir bestimmte Gruppen von Städten filtern, etwa Großstädte. Dazu können auch andere Datenquellen herangezogen werden, je nachdem welcher Filter gewünscht ist.

10 - Bundesländer

40 - Landkreise

50 - Gemeindeverbände

60 - Gemeinden

PR - Planungsregionen

id entspricht dem ARS Schlüssel

In [ ]:
big_cities = entities_df[(entities_df["population"] >= 100000)].copy()
big_cities = big_cities[big_cities["ris"].notnull()]

In [ ]:
big_cities = big_cities[big_cities["level"].isin(["60", "50", "10"])]
big_cities = big_cities[~big_cities["parents"].apply(lambda x: x[0]["name"] if x else None).isin(["Berlin, Stadt", "Hamburg, Freie und Hansestadt"])]

In [ ]:
big_cities.to_csv("./data/raw/big_cities.csv", index=False)